In [1]:
# !pip install sacremoses
# !pip install peft

In [2]:
import sys
import os
import pandas as pd
from datasets import Dataset
import torch
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

sys.path.append(os.path.abspath(".."))
from utils.preprocess import load_and_preprocess, upsample_no_and_maybe
from utils.llm_utils import LLMUtils
from utils.evaluation import print_metrics

In [3]:
llmUtils = LLMUtils(("microsoft/BioGPT"))

In [4]:
# Initializing the model and tokenizer
tokenizer = BioGptTokenizer.from_pretrained("microsoft/BioGPT")
model = BioGptForCausalLM.from_pretrained("microsoft/BioGPT")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

In [5]:
# Setting the padding end-of-sequence as padding token if padding token is not there
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [6]:
has_gpu = torch.cuda.is_available()

if has_gpu:
    torch.cuda.empty_cache()

device = torch.device("cuda" if has_gpu else "cpu") # My lap has GPU, but making it as dynamic, so you can run with cpu as well.
model.to(device)
model.eval()

BioGptForCausalLM(
  (biogpt): BioGptModel(
    (embed_tokens): BioGptScaledWordEmbedding(42384, 1024, padding_idx=1)
    (embed_positions): BioGptLearnedPositionalEmbedding(1026, 1024)
    (layers): ModuleList(
      (0-23): 24 x BioGptDecoderLayer(
        (self_attn): BioGptAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024

In [7]:
# Load data
train_df = load_and_preprocess('../Preprocessing+baseline/data/ori_pqal.json', False)
test_df = load_and_preprocess('../Preprocessing+baseline/data/test_set.json', False)

In [8]:
upsampled_df = upsample_no_and_maybe(test_df)

In [9]:
upsampled_df['label'].value_counts()

label
maybe    276
yes      276
no       276
Name: count, dtype: int64

In [10]:
# Tokenize the the training data
upsampled_df['tokens'] = upsampled_df.apply(llmUtils.tokenize_data, axis=1)
upsampled_df['tokens']

477    {'input_ids': [4790, 20925, 20, 10964, 10057, ...
457    {'input_ids': [4790, 20925, 20, 8928, 5305, 60...
462    {'input_ids': [4790, 20925, 20, 2882, 223, 14,...
174    {'input_ids': [4790, 20925, 20, 2882, 4970, 55...
484    {'input_ids': [4790, 20925, 20, 24235, 463, 9,...
                             ...                        
259    {'input_ids': [4790, 20925, 20, 10552, 23161, ...
32     {'input_ids': [4790, 20925, 20, 9260, 1632, 5,...
298    {'input_ids': [4790, 20925, 20, 8928, 14967, 1...
408    {'input_ids': [4790, 20925, 20, 2882, 1422, 24...
257    {'input_ids': [4790, 20925, 20, 10552, 60, 798...
Name: tokens, Length: 828, dtype: object

In [11]:
# Create dataframe from the tokens
tokenized_df = pd.DataFrame(upsampled_df["tokens"].tolist())
tokenized_df

,input_ids,attention_mask,labels
0,"[4790, 20925, 20, 10964, 10057, 1309, 17826, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,"[4790, 20925, 20, 8928, 5305, 600, 5, 23032, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,"[4790, 20925, 20, 2882, 223, 14, 151, 16, 7380...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,"[4790, 20925, 20, 2882, 4970, 5595, 24272, 126...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,"[4790, 20925, 20, 24235, 463, 9, 798, 28, 33, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...
823,"[4790, 20925, 20, 10552, 23161, 754, 5, 33651,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
824,"[4790, 20925, 20, 9260, 1632, 5, 5239, 204, 20...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
825,"[4790, 20925, 20, 8928, 14967, 13710, 8, 8545,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
826,"[4790, 20925, 20, 2882, 1422, 245, 1378, 14, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [12]:
# Create dataset from the tokenized dataframe
train_dataset = Dataset.from_pandas(tokenized_df[["input_ids", "attention_mask", "labels"]])
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 828
})

In [13]:
# Configuring LoRA to do parameter-efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 348,336,128 || trainable%: 0.4515


In [14]:
training_args = TrainingArguments(
    output_dir="./biogpt-lora-pubmedqa",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=has_gpu,
    report_to="none",
)

In [15]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]
    labels = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=custom_collate,
)

In [17]:
trainer.train()

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


Step,Training Loss
50,1.889448
100,1.481190
150,1.118834
200,1.494373
250,1.112628
300,1.012208
350,1.204473
400,1.089857
450,1.074342
500,0.879733


c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


TrainOutput(global_step=1242, training_loss=0.8883998029281935, metrics={'train_runtime': 225.4276, 'train_samples_per_second': 11.019, 'train_steps_per_second': 5.51, 'total_flos': 1576032651780096.0, 'train_loss': 0.8883998029281935, 'epoch': 3.0})

In [18]:
def generate_answer(question, context, max_new_tokens=20):
    prompt = llmUtils.build_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # extracting only the newly generated text from the full text
    generated_text = full_text[len(prompt):].strip()
    return generated_text

In [19]:
predicted_values = []
actual_values = []

In [20]:
for index, row in test_df.iterrows():
    # normalizing the labels of the test data
    actual_values.append(llmUtils.normalize_label(row["label"]))

    # generating and normalizing the labels of the generated data
    generated_answer = generate_answer(row["question"], row["context"])
    predicted_values.append(llmUtils.normalize_label(generated_answer))

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


In [21]:
print_metrics(actual_values, predicted_values)

Accuracy 0.638
Macro F1 0.4922829088481513
Classification_report

              precision    recall  f1-score   support

         yes       0.62      0.95      0.75       276
          no       0.72      0.28      0.40       169
       maybe       0.92      0.20      0.33        55

    accuracy                           0.64       500
   macro avg       0.75      0.47      0.49       500
weighted avg       0.69      0.64      0.58       500

